In [ ]:
# GeoProfiler
# Automated Topographic Line / Swath Profile Extraction Tool
# Developed by: Chandni Verma

# ===============================
# REQUIREMENTS / INPUT GUIDELINES
# ===============================

# 1. DEM Requirements:
#    - DEM should be in a Projected Coordinate Reference System (Projected CRS),
#      e.g., UTM
#    - Units should be in meters for accurate distance/elevation profiling
#    - Sink-filled DEM recommended for hydrologically corrected/smoother profiles
#    - Bilinear resampling suggested if DEM has been resampled

# 2. Vector Input Requirements:
#    - Input line shapefile can represent:
#         • River/stream centerline
#         • Transect line
#         • Fault/scarp profile line
#    - Required shapefile components:
#         .shp, .shx, .dbf, .prj

# 3. CRS Handling:
#    - If shapefile CRS differs from DEM CRS,
#      code automatically reprojects vector to DEM CRS

# 4. User Settings:
#    - profile_type = "line" → Longitudinal/Cross-sectional profile
#    - profile_type = "swath" → Swath profile (min/mean/max elevation envelope)
#    - swath_width is used only in swath mode

# 5. Output:
#    - Output plots are automatically saved in PNG and vector PDF formats


import rasterio
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import os
from scipy.ndimage import gaussian_filter1d
from shapely.geometry import LineString, MultiLineString

# ---------------------------------------------------
# CREATE OUTPUT FOLDER
# ---------------------------------------------------
os.makedirs("outputs", exist_ok=True)

# ---------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------
# Choose profile type:
# "line"  → longitudinal or cross-sectional profile
# "swath" → Min-Max and Mean Envelope

profile_type = "line"           # "line" or "swath"
swath_width = 600               # meters

# ---------------------------------------------------
# INPUT
# ---------------------------------------------------
dem_path = "/path/to/dem.tif"
line_path = "/path/to/line_shapefile.shp"

# ---------------------------------------------------
# CRS CHECK
# ---------------------------------------------------
src = rasterio.open(dem_path)
gdf = gpd.read_file(line_path)

print("Raster CRS:", src.crs)
print("Vector CRS:", gdf.crs)

if gdf.crs != src.crs:
    gdf = gdf.to_crs(src.crs)
    print("Reprojected vector to raster CRS")
    print("Vector CRS:", gdf.crs)

# ---------------------------------------------------
# PARAMETERS
# ---------------------------------------------------
dx = src.res[0]
dy = src.res[1]

# ---------------------------------------------------
# LOOP THROUGH EACH PROFILE
# ---------------------------------------------------
for idx, geom in enumerate(gdf.geometry):

    # HANDLE GEOMETRY
    if isinstance(geom, MultiLineString):
        line = max(geom.geoms, key=lambda g: g.length)
    elif isinstance(geom, LineString):
        line = geom
    else:
        continue

    print(f"Processing profile {idx+1}")

    # ---------------------------------------------------
    # SAMPLE CENTERLINE
    # ---------------------------------------------------
    distances = np.arange(0, line.length, dx)
    pts = [line.interpolate(d) for d in distances]

    # SKIP VERY SHORT LINES
    if len(pts) < 2:
        print(f"Skipping short profile {idx+1}")
        continue

    xy = [(p.x, p.y) for p in pts]

    center_elev = np.array([v[0] for v in src.sample(xy)]).astype(float)

    # NODATA
    nodata = src.nodata
    if nodata is not None:
        center_elev[center_elev == nodata] = np.nan

    # OUTLIER REMOVAL
    diff = np.abs(np.diff(center_elev, prepend=center_elev[0]))
    threshold = np.nanpercentile(diff, 95)
    center_elev[diff > threshold] = np.nan

    # INTERPOLATE
    nans = np.isnan(center_elev)
    if np.any(nans) and np.sum(~nans) > 1:
        center_elev[nans] = np.interp(
            np.flatnonzero(nans),
            np.flatnonzero(~nans),
            center_elev[~nans]
        )

    # SMOOTH
    smooth_elev = gaussian_filter1d(center_elev, sigma=5)

    dist_km = distances / 1000

    # ===================================================
    # LINE PROFILE
    # ===================================================
    if profile_type == "line":

        plt.figure(figsize=(8,3))
        plt.plot(dist_km, smooth_elev, color="blue", linewidth=1.2,
                 label="River/Cross-section")

        plt.xlabel("Distance (km)")
        plt.ylabel("Elevation (m)")
        plt.title(f"Topographic Line Profile – Line {idx+1}")
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            f"outputs/topographic_line_profile_line_{idx+1}.pdf",
            dpi=300,
            bbox_inches="tight"
            )

        plt.savefig(
            f"outputs/topographic_line_profile_line_{idx+1}.png",
            dpi=300,
            bbox_inches="tight"
            )

        plt.show()
        plt.close()

    # ===================================================
    # SWATH PROFILE
    # ===================================================
    elif profile_type == "swath":

        offsets = np.arange(-swath_width/2, swath_width/2 + dy, dy)
        all_elev = []

        # SAMPLE ACROSS NORMALS
        for off in offsets:
            swath_xy = []

            for i, p in enumerate(pts):

                # Compute tangent
                if i == 0:
                    p1 = pts[i]
                    p2 = pts[i+1]
                elif i == len(pts)-1:
                    p1 = pts[i-1]
                    p2 = pts[i]
                else:
                    p1 = pts[i-1]
                    p2 = pts[i+1]

                tx = p2.x - p1.x
                ty = p2.y - p1.y

                norm = np.sqrt(tx**2 + ty**2)
                if norm == 0:
                    swath_xy.append((p.x, p.y))
                    continue

                tx /= norm
                ty /= norm

                # Perpendicular normal
                nx = -ty
                ny = tx

                # Offset point
                x_off = p.x + off * nx
                y_off = p.y + off * ny

                swath_xy.append((x_off, y_off))

            elev = np.array([v[0] for v in src.sample(swath_xy)]).astype(float)

            if nodata is not None:
                elev[elev == nodata] = np.nan

            # OUTLIER REMOVAL
            diff = np.abs(np.diff(elev, prepend=elev[0]))
            threshold = np.nanpercentile(diff, 95)
            elev[diff > threshold] = np.nan

            # INTERPOLATE
            nans = np.isnan(elev)
            if np.any(nans) and np.sum(~nans) > 1:
                elev[nans] = np.interp(
                    np.flatnonzero(nans),
                    np.flatnonzero(~nans),
                    elev[~nans]
                )

            all_elev.append(elev)

        all_elev = np.array(all_elev)

        # STATS
        elev_min = np.nanmin(all_elev, axis=0)
        elev_mean = np.nanmean(all_elev, axis=0)
        elev_max = np.nanmax(all_elev, axis=0)

        # SMOOTH
        elev_min_smooth = gaussian_filter1d(elev_min, sigma=5)
        elev_mean_smooth = gaussian_filter1d(elev_mean, sigma=5)
        elev_max_smooth = gaussian_filter1d(elev_max, sigma=5)

        # PLOT
        plt.figure(figsize=(12,3))

        plt.fill_between(dist_km, elev_min_smooth, elev_max_smooth,
                         color="lightgray", alpha=0.8, label="Swath min-max")

        plt.plot(dist_km, elev_mean_smooth, color="black", linewidth=1,
                 label="Mean")

        plt.xlabel("Distance (km)")
        plt.ylabel("Elevation (m)")
        plt.title(f"Topographic Swath Profile – Line {idx+1}")
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            f"outputs/topographic_swath_profile_line_{idx+1}.pdf",
            dpi=300,
            bbox_inches="tight"
        )

        plt.savefig(
            f"outputs/topographic_swath_profile_line_{idx+1}.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

# ---------------------------------------------------
# CLOSE FILE
# ---------------------------------------------------
src.close()